# RSA — 01: Compute RDMs and Define Model RDMs

Computes neural representational dissimilarity matrices (RDMs) using the
crossnobis estimator, then builds theoretical model RDMs to compare against.

**Output:** per-subject RDM files saved to the analysis directory

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config
from eeg_toolkit import compute_rdms_all, load_all_rdms

# ── Update this path ──
cfg = load_config('../../configs/your_experiment.yaml')

print("Setup OK")

In [ ]:
# ── Define conditions for RSA ──
# Each key is a condition label; each value is a list of event codes.
# Use descriptive names — they will label the RDM axes.
# ── Update to match your experiment's event codes ──
conditions = {
    'cond_01': [10],
    'cond_02': [11],
    'cond_03': [12],
    'cond_04': [13],
    # add more conditions as needed
}

summary, group = compute_rdms_all(
    cfg,
    window_name='your_window',
    conditions=conditions,
    analysis_name='my_rsa',
    resample_sfreq=256,
    baseline=(-0.2, 0.0),
    method='crossnobis',
    n_jobs=1,
    overwrite=False,
)

In [ ]:
# ── Build theoretical model RDMs ──
# Each model encodes a hypothesis about what structure should be in the data.
# ── Update to reflect the dimensions/features of your stimuli ──
import numpy as np
from eeg_toolkit.rsa import square_to_vec

cond_names = sorted(conditions.keys())
n_cond = len(cond_names)

# Example: assign a feature value to each condition
# ── Replace with your own feature values ──
feature_values = np.array([1, 1, 2, 2])   # e.g. category labels

# Model 1: identity (binary — same vs different)
model_identity = (feature_values[:, None] != feature_values[None, :]).astype(float)

# Model 2: distance (graded — |val_i - val_j|)
model_distance = np.abs(feature_values[:, None] - feature_values[None, :]).astype(float)

models = {
    'Feature identity': model_identity,
    'Feature distance': model_distance,
}

# Check inter-model correlations
from scipy.stats import spearmanr
print("Model correlations:")
model_vecs = {k: square_to_vec(v) for k, v in models.items()}
names = list(model_vecs.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        r, _ = spearmanr(model_vecs[names[i]], model_vecs[names[j]])
        print(f"  {names[i]} × {names[j]}: r = {r:.3f}")

In [ ]:
# ── Visualize model RDMs ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
if len(models) == 1:
    axes = [axes]

for ax, (name, mat) in zip(axes, models.items()):
    im = ax.imshow(mat, cmap='viridis', aspect='equal')
    ax.set_title(name)
    ax.set_xticks(range(n_cond))
    ax.set_yticks(range(n_cond))
    ax.set_xticklabels(cond_names, rotation=90, fontsize=8)
    ax.set_yticklabels(cond_names, fontsize=8)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualize neural RDMs across time windows ──
import mne
from eeg_toolkit.rsa import load_all_rdms, vec_to_square

group = load_all_rdms(cfg, 'your_window', 'my_rsa')
times = group['times']
n_cond = group['n_conditions']

time_windows = [
    ('0–200 ms', 0.0, 0.2),
    ('200–400 ms', 0.2, 0.4),
    ('400–600 ms', 0.4, 0.6),
]

fig, axes = plt.subplots(1, len(time_windows), figsize=(5 * len(time_windows), 4))
for ax, (label, tmin, tmax) in zip(axes, time_windows):
    t_mask = (times >= tmin) & (times <= tmax)
    mean_vec = group['dissimilarities'][:, t_mask, :].mean(axis=(0, 1))
    rdm_mat = vec_to_square(mean_vec, n_cond)
    im = ax.imshow(rdm_mat, cmap='viridis', aspect='equal')
    ax.set_title(label)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()